# Leitor revisado v2 — reproduzir o ajuste já testado

O leitor escolhe qual fato da ficha responde à pergunta e separa perguntas sem resposta. Reutiliza integralmente o Transformer e o tokenizador próprios; ajusta **somente 385 parâmetros** da camada de decisão. Não pré-treina novamente. CPU é suficiente.

Os pesos novos já foram treinados e medidos; este caderno serve para reproduzir o experimento. No controle congelado v2, o leitor passou de **17 para 25 acertos em 48 perguntas respondíveis (35,42% → 52,08%)**; AUC **0,573 → 0,745**. No v1, **22 → 30 acertos em 50 (44% → 60%)**. Os assuntos e as perguntas desses controles não aparecem na supervisão.

Isso mede o leitor isolado. Ele permanece **experimental, sem aprovação para o chat**. O combinador do Crivo ainda precisa entregar mais respostas certas sem mais erros.

O tutor agora fornece supervisão. A seleção do ajuste usa cinco blocos externos e quatro internos separados por assunto. Os testes congelados são medidos só em agregados, depois de congelar os pesos, e não escolhem hiperparâmetros.


## 1. Preparar os pesos existentes e o Drive

O modo padrão **pesos_prontos** monta os pesos novos já treinados e medidos, sem repetir o treino ou os testes. Use **reproduzir_treino** somente para reproduzir o experimento completo. Execute as células na ordem. Os arquivos ficam em uma pasta nova, `CRIVO/leitor-revisado-v2`.


In [ ]:
MODO = 'pesos_prontos' # @param ["pesos_prontos", "reproduzir_treino"]
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile
drive.mount('/content/drive')
ROOT=Path('/content/CRIVO-leitor-revisado')
OUT=Path('/content/drive/MyDrive/CRIVO/leitor-revisado-v2')
OUT.mkdir(parents=True,exist_ok=True)
REVISAO='05a9fce4240d159813c0b138989d9ea4ff9fcb41'
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/HROSONE/CRIVO.git',str(ROOT)],check=True)
subprocess.run(['git','diff','--exit-code'],cwd=ROOT,check=True)
subprocess.run(['git','fetch','--depth','1','origin',REVISAO],cwd=ROOT,check=True)
subprocess.run(['git','checkout','--detach',REVISAO],cwd=ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','torch'],check=True)
os.environ['CRIVO_REPO']=str(ROOT)
os.environ['CRIVO_SAIDA']=str(OUT)
ORIGEM=ROOT/'experimentos/pesos_base/leitor_transformer'
assert hashlib.sha256((ORIGEM/'pesos_numpy.npz').read_bytes()).hexdigest()=='603ff3b6765d9fc82ff53517560ba3f1abca595e27f5a98c55911f1bb9a2b6dc'
assert hashlib.sha256((ORIGEM/'tokenizer.json').read_bytes()).hexdigest()=='6e37a7508def06d31f648a1fb816a67df99e806d450051b611a2406f044dbe7e'
print('Leitor próprio existente conferido. Este ajuste roda em CPU e treina somente 385 parâmetros.')


In [ ]:
CODIGOS={'continuar_leitor.py': '"""Ajuste contrastivo do leitor próprio já treinado; nunca pré-treina do zero."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\n\ndef sha256(caminho):\n    return hashlib.sha256(Path(caminho).read_bytes()).hexdigest()\n\n\ndef carregar(origem, dispositivo):\n    from linguagem_profunda import Configuracao, LinguagemProfunda\n    from pontuador_frases import BPE\n    origem = Path(origem)\n    meta = json.loads((origem / \'meta.json\').read_text(encoding=\'utf-8\'))\n    with np.load(origem / \'pesos_numpy.npz\', allow_pickle=False) as z:\n        config = json.loads(str(z[\'meta\'].item()))\n        chaves = (\'vocabulario\', \'dimensao\', \'camadas\', \'cabecas\', \'contexto\', \'dropout\')\n        cfg = Configuracao(**{k: config[k] for k in chaves})\n        if config[\'tokenizer_sha256\'] != sha256(origem / \'tokenizer.json\'):\n            raise ValueError(\'Tokenizador não corresponde aos pesos existentes.\')\n\n        class Leitor(nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.base = LinguagemProfunda(cfg)\n                self.cabeca = nn.Linear(cfg.dimensao, 1)\n\n            def forward(self, ids, ultimos):\n                b = self.base\n                x = b.embedding(ids) + b.posicao(torch.arange(ids.shape[1], device=ids.device))\n                for bloco in b.blocos:\n                    x = bloco(x)\n                x = b.norm(x)\n                return self.cabeca(x[torch.arange(len(ids), device=ids.device), ultimos]).squeeze(-1)\n\n        modelo = Leitor()\n        estado = {}\n        for nome, tensor in modelo.state_dict().items():\n            chave = nome[5:] if nome.startswith(\'base.\') else nome\n            if chave not in z or tuple(z[chave].shape) != tuple(tensor.shape):\n                raise ValueError(\'Peso ausente ou formato incompatível: \' + chave)\n            estado[nome] = torch.from_numpy(np.array(z[chave], dtype=np.float32, copy=True))\n        modelo.load_state_dict(estado, strict=True)\n    return modelo.to(dispositivo), BPE(origem / \'tokenizer.json\'), meta\n\n\ndef configurar_ajuste(modelo, camadas):\n    if not 1 <= camadas <= len(modelo.base.blocos):\n        raise ValueError(\'Número de camadas inválido.\')\n    for parametro in modelo.parameters():\n        parametro.requires_grad_(False)\n    for modulo in list(modelo.base.blocos[-camadas:]) + [modelo.base.norm, modelo.cabeca]:\n        for parametro in modulo.parameters():\n            parametro.requires_grad_(True)\n\n\ndef modo_treino(modelo, camadas):\n    modelo.train()\n    for bloco in modelo.base.blocos[:-camadas]:\n        bloco.eval()\n\n\ndef perda_grupo(logits, alvo):\n    # Fatos da MESMA ficha competem; zero representa "nenhum responde".\n    alvo_rank = len(logits) if alvo is None else alvo\n    ranking = torch.nn.functional.cross_entropy(\n        torch.cat([logits, logits.new_zeros(1)])[None],\n        torch.tensor([alvo_rank], device=logits.device))\n    y = torch.zeros_like(logits)\n    if alvo is not None:\n        y[alvo] = 1\n    binaria = torch.nn.functional.binary_cross_entropy_with_logits(\n        logits, y, pos_weight=logits.new_tensor(max(1, min(4, len(logits) - 1))))\n    return ranking + .5 * binaria\n\n\ndef atualizar(modelo, bpe, grupos, rng, opt, lote, camadas, dispositivo):\n    from scripts.treinar_leitor_transformer import lote_tensores\n    positivos, negativos = grupos\n    if not positivos or not negativos:\n        raise ValueError(\'Treino exige exemplos respondíveis e sem resposta.\')\n    amostra = [rng.choice(negativos if rng.random() < .25 else positivos) for _ in range(lote)]\n    pares, limites = [], []\n    for pergunta, fatos, alvo in amostra:\n        ini = len(pares)\n        pares.extend((pergunta, fato) for fato in fatos)\n        limites.append((ini, len(pares), alvo))\n    ids, ultimos = lote_tensores(pares, bpe, modelo.base.config.contexto, dispositivo)\n    modo_treino(modelo, camadas)\n    opt.zero_grad(set_to_none=True)\n    logits = modelo(ids, ultimos)\n    perda = torch.stack([perda_grupo(logits[a:b], alvo) for a, b, alvo in limites]).mean()\n    if not torch.isfinite(perda):\n        raise RuntimeError(\'Perda não finita; treino interrompido.\')\n    perda.backward()\n    torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.)\n    opt.step()\n    return float(perda.detach())\n\n\ndef gravar_checkpoint(caminho, modelo, opt, rng, passo, assinatura, melhor):\n    caminho = Path(caminho)\n    caminho.parent.mkdir(parents=True, exist_ok=True)\n    estado = {\'modelo\': modelo.state_dict(), \'otimizador\': opt.state_dict(),\n              \'rng_python\': rng.getstate(), \'rng_torch\': torch.get_rng_state(),\n              \'rng_cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n              \'passo\': passo, \'assinatura\': assinatura, \'melhor\': melhor}\n    temporario = caminho.with_suffix(\'.tmp\')\n    torch.save(estado, temporario)\n    temporario.replace(caminho)\n\n\ndef retomar(caminho, modelo, opt, rng, assinatura):\n    estado = torch.load(caminho, map_location=\'cpu\', weights_only=True)\n    if estado[\'assinatura\'] != assinatura:\n        raise ValueError(\'Checkpoint de outro código, corpus, pesos ou configuração. Use outra pasta.\')\n    modelo.load_state_dict(estado[\'modelo\'], strict=True)\n    opt.load_state_dict(estado[\'otimizador\'])\n    rng.setstate(estado[\'rng_python\'])\n    torch.set_rng_state(estado[\'rng_torch\'])\n    if estado[\'rng_cuda\']:\n        if not torch.cuda.is_available():\n            raise ValueError(\'Checkpoint de GPU exige GPU para a continuação.\')\n        torch.cuda.set_rng_state_all(estado[\'rng_cuda\'])\n    return estado[\'passo\'], estado[\'melhor\']\n\n\ndef exportar(modelo, origem, saida, meta_original, passo, metricas, baseline):\n    saida = Path(saida)\n    saida.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(Path(origem) / \'tokenizer.json\', saida / \'tokenizer.json\')\n    config = vars(modelo.base.config).copy()\n    config.update(passo=passo, tokenizer_sha256=sha256(saida / \'tokenizer.json\'))\n    pesos = {k: v.detach().float().cpu().numpy().astype(np.float16)\n             for k, v in modelo.base.state_dict().items()}\n    pesos[\'cabeca.weight\'] = modelo.cabeca.weight.detach().float().cpu().numpy()\n    pesos[\'cabeca.bias\'] = modelo.cabeca.bias.detach().float().cpu().numpy()\n    np.savez_compressed(saida / \'pesos_numpy.npz\', meta=np.array(json.dumps(config)), **pesos)\n    meta = json.loads(json.dumps(meta_original))\n    meta[\'validacao_tutor\'] = metricas\n    meta[\'ajuste_contrastivo\'] = {\'passos_adicionais\': passo, \'baseline_tutor\': baseline,\n                                \'origem_sha256\': sha256(Path(origem) / \'pesos_numpy.npz\'),\n                                \'objetivo\': \'ranking de fatos da mesma ficha e nenhuma resposta\'}\n    meta[\'controle\'] = {\'aprovado\': False, \'criterio\': \'Requer ganho no controle e testes congelados.\'}\n    (saida / \'meta.json\').write_text(json.dumps(meta, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--repo\', required=True)\n    ap.add_argument(\'--origem\', required=True)\n    ap.add_argument(\'--saida\', required=True)\n    ap.add_argument(\'--checkpoint\', required=True)\n    ap.add_argument(\'--passos\', type=int, default=1000)\n    ap.add_argument(\'--lote\', type=int, default=4, help=\'Perguntas por passo, com todos os fatos de cada ficha\')\n    ap.add_argument(\'--camadas\', type=int, default=2)\n    ap.add_argument(\'--lr\', type=float, default=1e-5)\n    ap.add_argument(\'--avaliar-a-cada\', type=int, default=200)\n    ap.add_argument(\'--checkpoint-a-cada\', type=int, default=100)\n    ap.add_argument(\'--max-horas\', type=float, default=1.5)\n    ap.add_argument(\'--dispositivo\', default=\'cuda\')\n    args = ap.parse_args()\n    if min(args.passos, args.lote, args.avaliar_a_cada, args.checkpoint_a_cada) <= 0:\n        ap.error(\'Passos, lote e intervalos devem ser positivos.\')\n    if args.dispositivo == \'cuda\' and not torch.cuda.is_available():\n        ap.error(\'Selecione GPU no Colab.\')\n    sys.path.insert(0, str(Path(args.repo).resolve()))\n    sys.path.insert(0, str(Path(args.repo).resolve() / \'scripts\'))\n    from scripts.treinar_leitor_transformer import exemplos_sinteticos, exemplos_tutor, avaliar\n    from crivo import Crivo\n    torch.set_num_threads(2)\n    torch.manual_seed(20261008)\n    comp = Crivo().compositor\n    grupos = exemplos_sinteticos(comp, random.Random(20261008))\n    validacao = exemplos_tutor(comp)\n    # Usa os mesmos assuntos excluídos do treinador original; tutor/congelados nunca treinam.\n    modelo, bpe, meta_original = carregar(args.origem, args.dispositivo)\n    configurar_ajuste(modelo, args.camadas)\n    opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=args.lr, weight_decay=.01)\n    rng = random.Random(20261008)\n    corpus_sha = hashlib.sha256(json.dumps([grupos, validacao], ensure_ascii=False).encode()).hexdigest()\n    assinatura = {\'origem\': sha256(Path(args.origem) / \'pesos_numpy.npz\'),\n                  \'tokenizador\': sha256(Path(args.origem) / \'tokenizer.json\'),\n                  \'codigo\': sha256(__file__), \'corpus\': corpus_sha,\n                  \'lr\': args.lr, \'lote\': args.lote, \'camadas\': args.camadas,\n                  \'dispositivo\': args.dispositivo, \'torch\': str(torch.__version__),\n                  \'modelo\': sha256(Path(args.repo) / \'linguagem_profunda.py\'),\n                  \'sequencia\': sha256(Path(args.repo) / \'leitor_transformer.py\'),\n                  \'treinador_base\': sha256(Path(args.repo) / \'scripts/treinar_leitor_transformer.py\')}\n    grupos = ([g for g in grupos if g[2] is not None], [g for g in grupos if g[2] is None])\n    saida = Path(args.saida)\n    baseline_path = saida / \'baseline.json\'\n    saida.mkdir(parents=True, exist_ok=True)\n    checkpoint = Path(args.checkpoint)\n    if checkpoint.exists():\n        if not baseline_path.exists() or not (saida / \'melhor\' / \'pesos_numpy.npz\').exists():\n            raise ValueError(\'Checkpoint sem baseline ou melhor candidato; restaure a pasta completa.\')\n        baseline = json.loads(baseline_path.read_text())\n        passo, melhor = retomar(checkpoint, modelo, opt, rng, assinatura)\n    else:\n        baseline = avaliar(modelo, bpe, modelo.base.config.contexto, validacao, args.dispositivo)\n        baseline_path.write_text(json.dumps(baseline, indent=2) + \'\\n\')\n        passo, melhor = 0, baseline\n        exportar(modelo, args.origem, saida / \'melhor\', meta_original, 0, baseline, baseline)\n    if passo > args.passos:\n        raise ValueError(\'O checkpoint já ultrapassou o horizonte solicitado.\')\n    print(\'Base existente:\', meta_original[\'base\'][\'passo_pretreino\'], \'passos de pré-treino; mantida.\', flush=True)\n    print(\'Baseline tutor:\', baseline, \'| retomando ajuste no passo\', passo, flush=True)\n    inicio, passo_inicial = time.monotonic(), passo\n    tempo_treino = 0.\n    while passo < args.passos:\n        inicio_passo = time.monotonic()\n        perda = atualizar(modelo, bpe, grupos, rng, opt, args.lote, args.camadas, args.dispositivo)\n        tempo_treino += time.monotonic() - inicio_passo\n        passo += 1\n        fim_tempo = args.max_horas > 0 and time.monotonic() - inicio >= args.max_horas * 3600\n        if passo % 20 == 0:\n            seg = tempo_treino / max(1, passo - passo_inicial)\n            print(\'passo %d/%d perda %.4f | %.2fs/passo | treino restante ~%.1fmin (avaliações à parte)\' %\n                  (passo, args.passos, perda, seg, (args.passos - passo) * seg / 60), flush=True)\n        if passo % args.avaliar_a_cada == 0 or passo == args.passos or fim_tempo:\n            metricas = avaliar(modelo, bpe, modelo.base.config.contexto, validacao, args.dispositivo)\n            print(\'Validação tutor:\', metricas, flush=True)\n            sem_regressao = all(metricas[k] >= baseline[k] for k in (\'acerto_fato\', \'auc_respondivel\'))\n            if sem_regressao and sum(metricas[k] for k in (\'acerto_fato\', \'auc_respondivel\')) > sum(\n                    melhor[k] for k in (\'acerto_fato\', \'auc_respondivel\')):\n                melhor = metricas\n                exportar(modelo, args.origem, saida / \'melhor\', meta_original, passo, metricas, baseline)\n        if passo % args.checkpoint_a_cada == 0 or passo == args.passos or fim_tempo:\n            gravar_checkpoint(checkpoint, modelo, opt, rng, passo, assinatura, melhor)\n            print(\'Checkpoint único gravado:\', passo, flush=True)\n        if fim_tempo:\n            print(\'Pausado por tempo; execute esta célula novamente para continuar.\', flush=True)\n            break\n    relatorio = {\'baseline\': baseline, \'melhor_tutor\': melhor, \'passos_adicionais\': passo,\n                 \'concluido\': passo == args.passos, \'aprovado\': False, \'assinatura\': assinatura}\n    (saida / \'relatorio.json\').write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + \'\\n\')\n    print(json.dumps(relatorio, ensure_ascii=False), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'diagnosticar_cabeca.py': 'import os\n"""Controle por assunto para adaptação apenas da cabeça do leitor próprio.\n\nO tutor é agora supervisão: nunca anunciar sua pontuação de treino como teste.\nOs cinco blocos externos recebem previsões de cabeças que não viram seu assunto.\nRegularização escolhida em quatro blocos internos, também separados por assunto.\n"""\nimport sys,json,hashlib,time,random\nfrom pathlib import Path\nimport numpy as np\nimport torch\nROOT=Path(os.environ[\'CRIVO_REPO\'])\nOUT=Path(os.environ[\'CRIVO_SAIDA\'])\nsys.path[:0]=[str(ROOT),str(ROOT/\'scripts\'),str(Path(__file__).resolve().parent)]\nfrom continuar_leitor import carregar,exportar\nfrom scripts.treinar_leitor_transformer import lote_tensores\n\ndef metricas(pred,grupos):\n    certos=0;pos=[];neg=[]\n    for z,c in zip(pred,grupos):\n        a=c[\'fato\'];k=int(np.argmax(z));p=float(torch.sigmoid(torch.tensor(z[k])))\n        if a is None:neg.append(p)\n        else:pos.append(p);certos+=k==a\n    auc=sum((a>b)+.5*(a==b) for a in pos for b in neg)/max(1,len(pos)*len(neg))\n    return dict(acerto_fato=certos/max(1,len(pos)),auc_respondivel=auc,certos=certos,respondiveis=len(pos),sem_resposta=len(neg))\n\ndef treinar(xs,cs,indices,w0,b0,l2):\n    # CE agrupada com opção fixa nenhuma + BCE; não mistura grupos de fichas.\n    w=torch.tensor(w0,requires_grad=True);b=torch.tensor(b0,requires_grad=True)\n    opt=torch.optim.LBFGS([w,b],lr=.5,max_iter=80,line_search_fn=\'strong_wolfe\')\n    x=torch.cat([xs[i] for i in indices]);ys=[];ranges=[];start=0\n    for i in indices:\n        n=len(xs[i]);a=cs[i][\'fato\'];y=torch.zeros(n)\n        if a is not None:y[a]=1\n        ys.append(y);ranges.append((start,start+n,n if a is None else a));start+=n\n    y=torch.cat(ys);group_weights=torch.cat([torch.full((len(xs[i]),),1/len(xs[i])) for i in indices])\n    init=torch.tensor(w0)\n    def closure():\n        opt.zero_grad();z=x@w+b\n        rank=torch.stack([torch.nn.functional.cross_entropy(torch.cat((z[a:e],z.new_zeros(1)))[None],torch.tensor([target])) for a,e,target in ranges]).mean()\n        binary=(torch.nn.functional.binary_cross_entropy_with_logits(z,y,reduction=\'none\')*group_weights).sum()/len(indices)\n        loss=rank+.5*binary+l2*((w-init).square().sum()+(b-b0).square())\n        loss.backward();return loss\n    opt.step(closure)\n    return w.detach(),b.detach()\n\ndef main():\n    torch.set_num_threads(2);torch.manual_seed(20261008)\n    cases=json.loads((ROOT/\'dados/leitura_ficha_tutor.json\').read_text())[\'casos\']\n    from crivo import Crivo\n    comp=Crivo().compositor\n    cases=[c for c in cases if c[\'assunto\'] in comp.itens]\n    origem=ROOT/\'experimentos/pesos_base/leitor_transformer\'\n    m,bpe,meta=carregar(origem,\'cpu\');m.eval()\n    cache=OUT/\'ocultos_tutor.npz\'\n    corpus=[dict(c,fatos=[f[\'texto\'] for f in comp.itens[c[\'assunto\']][\'fatos\']]) for c in cases]\n    sources={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in [ROOT/\'linguagem_profunda.py\',ROOT/\'leitor_transformer.py\',origem/\'tokenizer.json\']}\n    signature=hashlib.sha256((origem/\'pesos_numpy.npz\').read_bytes()+json.dumps([corpus,sources,str(torch.__version__)],sort_keys=True).encode()).hexdigest()\n    if cache.exists():\n        z=np.load(cache,allow_pickle=False)\n        if str(z[\'assinatura\'])!=signature:raise ValueError(\'Cache incompatível\')\n        xs=[torch.from_numpy(z[\'x\'+str(i)]) for i in range(len(cases))]\n    else:\n        xs=[]\n        with torch.no_grad():\n            for i,c in enumerate(cases):\n                fatos=[f[\'texto\'] for f in comp.itens[c[\'assunto\']][\'fatos\']]\n                ids,u=lote_tensores([(c[\'pergunta\'],f) for f in fatos],bpe,256,\'cpu\')\n                base=m.base;x=base.embedding(ids)+base.posicao(torch.arange(ids.shape[1]))\n                for bloco in base.blocos:x=bloco(x)\n                h=base.norm(x)[torch.arange(len(ids)),u].detach().cpu()\n                xs.append(h)\n                if i%25==0:print(\'extração\',i,\'/\',len(cases),flush=True)\n        np.savez_compressed(cache,assinatura=np.array(signature),**{\'x\'+str(i):x.numpy() for i,x in enumerate(xs)})\n    w0=m.cabeca.weight.detach()[0].numpy().copy();b0=float(m.cabeca.bias.detach()[0])\n    subjects=sorted({c[\'assunto\'] for c in cases});random.Random(20261008).shuffle(subjects)\n    fold={s:i%5 for i,s in enumerate(subjects)}\n    pred0=[(x@torch.tensor(w0)+b0).numpy() for x in xs]\n    print(\'baseline\',metricas(pred0,cases),flush=True)\n    # Grade estabelecida antes de observar quaisquer resultados externos.\n    grade=[.03,.1,.3,1.,3.];pred=[None]*len(cases);selection=[]\n    for outer in range(5):\n        train=[i for i,c in enumerate(cases) if fold[c[\'assunto\']]!=outer]\n        test=[i for i,c in enumerate(cases) if fold[c[\'assunto\']]==outer]\n        inner_subjects=[s for s in subjects if fold[s]!=outer]\n        inner={s:i%4 for i,s in enumerate(inner_subjects)}\n        scores=[]\n        for reg in grade:\n            ip=[];ic=[]\n            for k in range(4):\n                tr=[i for i in train if inner[cases[i][\'assunto\']]!=k]\n                va=[i for i in train if inner[cases[i][\'assunto\']]==k]\n                w,b=treinar(xs,cases,tr,w0,b0,reg)\n                ip.extend([(xs[i]@w+b).numpy() for i in va]);ic.extend([cases[i] for i in va])\n            met=metricas(ip,ic);scores.append((met[\'acerto_fato\']+met[\'auc_respondivel\'],reg,met))\n        _,reg,inner_met=max(scores,key=lambda r:(r[0],r[1]))\n        w,b=treinar(xs,cases,train,w0,b0,reg)\n        for i in test:pred[i]=(xs[i]@w+b).numpy()\n        selection.append({\'bloco\':outer,\'l2\':reg,\'validacao_interna\':inner_met,\'assuntos_externos\':[s for s in subjects if fold[s]==outer]})\n        print(\'bloco\',outer,\'l2\',reg,\'teste\',metricas([pred[i] for i in test],[cases[i] for i in test]),flush=True)\n    report={\'protocolo\':\'5 blocos externos por assunto; seleção L2 em 4 blocos internos; somente cabeça linear adaptada; tutor agora supervisão; testes congelados não usados\',\'baseline\':metricas(pred0,cases),\'validacao_externa\':metricas(pred,cases),\'selecao\':selection,\'origem_sha256\':hashlib.sha256((origem/\'pesos_numpy.npz\').read_bytes()).hexdigest(),\'corpus_sha256\':hashlib.sha256(json.dumps(corpus,sort_keys=True).encode()).hexdigest(),\'fontes\':sources,\'execucao\':{\'torch\':str(torch.__version__),\'numpy\':str(np.__version__),\'dispositivo\':\'cpu\',\'semente\':20261008},\'aprovado\':False}\n    wins=losses=0\n    for c,a,b in zip(cases,pred0,pred):\n        if c[\'fato\'] is None:continue\n        old=int(a.argmax())==c[\'fato\'];new=int(b.argmax())==c[\'fato\']\n        wins+=new and not old;losses+=old and not new\n    report[\'pares_acerto\']={\'ganhos\':wins,\'perdas\':losses}\n    (OUT/\'controle_cabeca.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\')\n    (OUT/\'predicoes_externas.json\').write_text(json.dumps([{\'assunto\':c[\'assunto\'],\'alvo\':c[\'fato\'],\'baseline\':a.tolist(),\'candidato\':b.tolist()} for c,a,b in zip(cases,pred0,pred)]))\n    print(json.dumps(report,ensure_ascii=False),flush=True)\n    # A regularização final usa somente a seleção interna, nunca resultados externos.\n    reg=float(np.median([r[\'l2\'] for r in selection]));w,b=treinar(xs,cases,list(range(len(cases))),w0,b0,reg)\n    with torch.no_grad():m.cabeca.weight.copy_(w[None]);m.cabeca.bias.copy_(b[None])\n    exportar(m,origem,OUT/\'candidato_cabeca\',meta,0,report[\'validacao_externa\'],report[\'baseline\'])\n    candidate_meta=json.loads((OUT/\'candidato_cabeca/meta.json\').read_text())\n    candidate_meta.pop(\'ajuste_contrastivo\',None)\n    candidate_meta[\'adaptacao_cabeca\']={\'parametros_treinados\':385,\'l2\':reg,\'dados\':\'tutor como supervisão, validação cruzada aninhada por assunto; base totalmente congelada\',\'controle\':report}\n    candidate_meta[\'controle\']={\'aprovado\':False,\'criterio\':\'Requer ganho end-to-end sem mais erros nos controles congelados.\'}\n    (OUT/\'candidato_cabeca/meta.json\').write_text(json.dumps(candidate_meta,ensure_ascii=False,indent=2)+\'\\n\')\n\nif __name__==\'__main__\':main()\n', 'controlar_pipeline.py': 'import os\n"""Validação do combinador com cabeças próprias treinadas sem o assunto externo."""\nimport sys,json,random\nfrom pathlib import Path\nimport numpy as np\nimport torch\nROOT=Path(os.environ[\'CRIVO_REPO\']);OUT=Path(os.environ[\'CRIVO_SAIDA\'])\nsys.path[:0]=[str(ROOT),str(ROOT/\'scripts\'),str(OUT),str(Path(__file__).resolve().parent)]\nfrom diagnosticar_cabeca import treinar as ajustar_cabeca\nfrom continuar_leitor import carregar\nfrom treinar_leitura_ficha import exemplos,treinar,propostas,politica,limiares,contar,regra,_entregues,precisao\nfrom leitura_ficha import LeituraFicha,TRACOS\n\ndef resumo(lines):\n    a,b=limiares(lines);aa,bb,cc=(contar(g) for g in politica(lines,a,b))\n    return {\'limiar\':a,\'limiar_aproximar\':b,\'validacao_cruzada_por_assunto\':{\'modelo_afirma\':aa,\'modelo_aproxima\':bb,\'modelo_cala\':cc},\'controle\':{\'aprovado\':False}}\n\ndef main():\n    torch.set_num_threads(2)\n    from crivo import Crivo\n    comp=Crivo().compositor\n    raw=json.loads((ROOT/\'dados/leitura_ficha_tutor.json\').read_text())[\'casos\']\n    raw=[c for c in raw if c[\'assunto\'] in comp.itens]\n    index={c[\'pergunta\']:i for i,c in enumerate(raw)}\n    path=OUT/\'tracos_tutor.json\'\n    if path.exists():cases=json.loads(path.read_text())\n    else:\n        cases,ignored=exemplos(comp,LeituraFicha(comp,caminho_modelo=\'/nao/existe\'))\n        path.write_text(json.dumps(cases,ensure_ascii=False)+\'\\n\')\n        print(\'casos elegíveis\',len(cases),\'ignorados\',ignored,flush=True)\n    z=np.load(OUT/\'ocultos_tutor.npz\',allow_pickle=False)\n    xs=[torch.from_numpy(z[\'x\'+str(i)]) for i in range(len(raw))]\n    m,_,_=carregar(ROOT/\'experimentos/pesos_base/leitor_transformer\',\'cpu\')\n    w0=m.cabeca.weight.detach()[0].numpy().copy();b0=float(m.cabeca.bias.detach()[0])\n    subjects=sorted({c[\'assunto\'] for c in raw});random.Random(20261008).shuffle(subjects)\n    fold={s:i%5 for i,s in enumerate(subjects)}\n    sel=json.loads((OUT/\'controle_cabeca.json\').read_text())[\'selecao\']\n    outputs={k:[] for k in [\'sem\',\'original\',\'adaptado\']}\n    for k in range(5):\n        train=[i for i,c in enumerate(raw) if fold[c[\'assunto\']]!=k]\n        w,b=ajustar_cabeca(xs,raw,train,w0,b0,sel[k][\'l2\'])\n        for name in outputs:\n            enriched=[]\n            for c in cases:\n                i=index[c[\'pergunta\']]\n                ps=torch.sigmoid(xs[i]@(torch.tensor(w0) if name==\'original\' else w)+(b0 if name==\'original\' else b)).tolist() if name!=\'sem\' else None\n                enriched.append(dict(c,x=[list(x)+([ps[j]] if ps else []) for j,x in enumerate(c[\'x\'])]))\n            tr=[c for c in enriched if fold[c[\'assunto\']]!=k];va=[c for c in enriched if fold[c[\'assunto\']]==k]\n            cw=treinar(tr)\n            outputs[name]+=propostas(va,lambda x:float(1/(1+np.exp(-(np.array(x)@cw)))))\n        print(\'combinador bloco\',k,\'feito\',flush=True)\n    result={k:resumo(lines) for k,lines in outputs.items()}\n    print(\'PIPELINE\',json.dumps(result,ensure_ascii=False),flush=True)\n    (OUT/\'controle_pipeline.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2)+\'\\n\')\n    # Modelo final em todos os dados de supervisão: não usar para alegar CV.\n    cm,_,_=carregar(OUT/\'candidato_cabeca\',\'cpu\')\n    final=[]\n    for c in cases:\n        ps=torch.sigmoid(cm.cabeca(xs[index[c[\'pergunta\']]])).squeeze(-1).detach().tolist()\n        final.append(dict(c,x=[list(x)+[ps[j]] for j,x in enumerate(c[\'x\'])]))\n    weights=treinar(final)\n    final_meta=dict(result[\'adaptado\'],versao=1,tracos=list(TRACOS)+[\'transformer\'],pesos=[round(float(v),5) for v in weights],dados={\'casos\':len(cases),\'supervisao\':\'tutor; leitor treinado dentro de cada bloco por assunto\'},comparacao=result)\n    (OUT/\'combinador_cabeca.json\').write_text(json.dumps(final_meta,ensure_ascii=False,indent=2)+\'\\n\')\n\nif __name__==\'__main__\':main()\n', 'avaliar_controles.py': 'import os\n"""Controles congelados: publica somente agregados, nunca perguntas ou erros."""\nimport sys,json,time,hashlib\nfrom pathlib import Path\nimport torch\nROOT=Path(os.environ[\'CRIVO_REPO\']);OUT=Path(os.environ[\'CRIVO_SAIDA\'])\nsys.path[:0]=[str(ROOT),str(ROOT/\'scripts\'),str(OUT),str(Path(__file__).resolve().parent)]\nfrom continuar_leitor import carregar\nfrom scripts.treinar_leitor_transformer import lote_tensores\nfrom diagnosticar_cabeca import metricas\n\ndef main():\n    torch.set_num_threads(2)\n    from crivo import Crivo\n    comp=Crivo().compositor\n    candidates={\'original\':ROOT/\'experimentos/pesos_base/leitor_transformer\',\'cabeca\':OUT/\'candidato_cabeca\'}\n    if (OUT/\'candidato_duas_camadas/pesos_numpy.npz\').exists():candidates[\'duas_camadas\']=OUT/\'candidato_duas_camadas\'\n    # Compromisso dos candidatos anterior à primeira leitura dos controles.\n    manifest={k:hashlib.sha256((v/\'pesos_numpy.npz\').read_bytes()).hexdigest() for k,v in candidates.items()}\n    (OUT/\'candidatos_antes_controle.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    report={\'protocolo\':\'Pesos congelados antes de abrir controles; avaliação só em agregados; não usados para otimização\',\'sha256\':manifest,\'conjuntos\':{},\'aprovado_chat\':False}\n    for name,path in [(\'v1\',ROOT/\'avaliacoes/leitura_ficha_v1/teste.json\'),(\'v2\',ROOT/\'avaliacoes/leitura_ficha_v2/teste.json\')]:\n        cases=json.loads(path.read_text())[\'casos\']\n        available=[c for c in cases if c[\'assunto\'] in comp.itens]\n        groups={}\n        for key,folder in candidates.items():\n            m,bpe,_=carregar(folder,\'cpu\');m.eval();pred=[]\n            with torch.no_grad():\n                for c in available:\n                    facts=[f[\'texto\'] for f in comp.itens[c[\'assunto\']][\'fatos\']]\n                    ids,u=lote_tensores([(c[\'pergunta\'],f) for f in facts],bpe,256,\'cpu\')\n                    pred.append(m(ids,u).numpy())\n            groups[key]=metricas(pred,available)\n            print(name,key,groups[key],flush=True)\n        report[\'conjuntos\'][name]={\'casos\':len(cases),\'avaliados\':len(available),\'metricas\':groups}\n        (OUT/\'controle_congelado.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\')\n    print(json.dumps(report,ensure_ascii=False),flush=True)\n\nif __name__==\'__main__\':main()\n', 'avaliar_chat.py': 'import os\n"""Comparação isolada do chat factual; pesos experimentais nunca são promovidos."""\nimport sys,json,torch\nfrom pathlib import Path\nROOT=Path(os.environ[\'CRIVO_REPO\']);OUT=Path(os.environ[\'CRIVO_SAIDA\'])\nsys.path[:0]=[str(ROOT),str(ROOT/\'scripts\'),str(Path(__file__).resolve().parent)]\nfrom continuar_leitor import carregar\nfrom scripts.treinar_leitor_transformer import lote_tensores\n\ndef main():\n    torch.set_num_threads(2)\n    import crivo\n    from leitura_ficha import LeituraFicha\n    from avaliar_leitura_ficha import avaliar\n    original=crivo.Crivo\n    m,bpe,_=carregar(OUT/\'candidato_cabeca\',\'cpu\');m.eval()\n    meta=json.loads((OUT/\'combinador_cabeca.json\').read_text())\n    class LeitorExperimental:\n        disponivel=True\n        def probabilidades(self,q,fatos):\n            if not fatos:return []\n            ids,u=lote_tensores([(q,f) for f in fatos],bpe,256,\'cpu\')\n            with torch.no_grad():return torch.sigmoid(m(ids,u)).tolist()\n    class Experimento(original):\n        def __init__(self,*args,**kwargs):\n            kwargs[\'usar_geracao\']=False\n            super().__init__(*args,**kwargs)\n            if modo==\'adaptado\':\n                reader=LeituraFicha(self.compositor,caminho_modelo=\'/nao/existe\',transformer=LeitorExperimental())\n                # Uso explícito no laboratório; arquivos continuam aprovado:false.\n                reader.pesos=meta[\'pesos\'];reader.limiar=meta[\'limiar\'];reader.limiar_aproximar=meta[\'limiar_aproximar\']\n                self._leitura_ficha=reader\n    result={\'protocolo\':\'Chat factual sem geração, classificadores originais, conjuntos congelados somente em agregados; instalação experimental em memória\',\'aprovado\':False,\'modos\':{}}\n    try:\n        crivo.Crivo=Experimento\n        for modo in [\'original\',\'adaptado\']:\n            result[\'modos\'][modo]={}\n            for conjunto in [\'teste\',\'teste_v2\']:\n                agg,_=avaliar(conjunto)\n                result[\'modos\'][modo][conjunto]=agg\n                print(modo,conjunto,json.dumps(agg,ensure_ascii=False),flush=True)\n                (OUT/\'controle_chat.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2)+\'\\n\')\n    finally:crivo.Crivo=original\n    print(json.dumps(result,ensure_ascii=False),flush=True)\n\nif __name__==\'__main__\':main()\n', 'ajustar_perguntas_revisadas.py': 'import os\n"""Experimento de adaptação das duas últimas camadas com perguntas revisadas.\n\nSomente desenvolvimento por assunto; testes congelados ficam fora de todo treino.\nCache determinístico das seis camadas congeladas evita repetir seu cálculo.\n"""\nimport sys,json,random,time,copy\nfrom pathlib import Path\nimport numpy as np\nimport torch\nROOT=Path(os.environ[\'CRIVO_REPO\']);OUT=Path(os.environ[\'CRIVO_SAIDA\'])\nsys.path[:0]=[str(ROOT),str(ROOT/\'scripts\'),str(OUT),str(Path(__file__).resolve().parent)]\nfrom continuar_leitor import carregar,configurar_ajuste,perda_grupo,exportar\nfrom scripts.treinar_leitor_transformer import lote_tensores\nfrom diagnosticar_cabeca import metricas\n\ndef predizer(m,cache,indices):\n    m.eval();out=[]\n    with torch.no_grad():\n        for i in indices:\n            x,u=cache[i];h=x\n            for block in m.base.blocos[-2:]:h=block(h)\n            out.append(m.cabeca(m.base.norm(h)[torch.arange(len(h)),u]).squeeze(-1).numpy())\n    return out\n\ndef treinar(m,cache,cases,train,dev,steps=200):\n    configurar_ajuste(m,2)\n    rng=random.Random(20261008);torch.manual_seed(20261008)\n    opt=torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],lr=5e-5,weight_decay=.01)\n    selected={k:v.detach().clone() for k,v in m.state_dict().items() if k.startswith((\'base.blocos.6.\',\'base.blocos.7.\',\'base.norm.\',\'cabeca.\'))}\n    baseline=metricas(predizer(m,cache,dev),[cases[i] for i in dev]) if dev else None\n    best=(baseline[\'acerto_fato\']+baseline[\'auc_respondivel\']) if baseline else -1\n    pos=[i for i in train if cases[i][\'fato\'] is not None];neg=[i for i in train if cases[i][\'fato\'] is None]\n    chosen_step=0;history=[];start=time.monotonic()\n    for step in range(1,steps+1):\n        sample=[rng.choice(neg if rng.random()<.25 else pos) for _ in range(8)]\n        total=sum(len(cache[i][0]) for i in sample);t=max(cache[i][0].shape[1] for i in sample)\n        h=torch.zeros((total,t,m.base.config.dimensao));last=torch.empty(total,dtype=torch.long);ranges=[];offset=0\n        for i in sample:\n            x,u=cache[i];n=len(x);h[offset:offset+n,:x.shape[1]]=x;last[offset:offset+n]=u\n            ranges.append((offset,offset+n,cases[i][\'fato\']));offset+=n\n        m.train();opt.zero_grad(set_to_none=True)\n        for block in m.base.blocos[-2:]:h=block(h)\n        z=m.cabeca(m.base.norm(h)[torch.arange(total),last]).squeeze(-1)\n        loss=torch.stack([perda_grupo(z[a:b],alvo) for a,b,alvo in ranges]).mean()\n        if not torch.isfinite(loss):raise RuntimeError(\'Perda não finita\')\n        loss.backward();torch.nn.utils.clip_grad_norm_([p for p in m.parameters() if p.requires_grad],1.)\n        opt.step()\n        if step%25==0 or step==steps:\n            met=metricas(predizer(m,cache,dev),[cases[i] for i in dev]) if dev else None\n            print(\'passo\',step,\'perda\',round(float(loss.detach()),4),\'dev\',met,\'segundos\',round(time.monotonic()-start,1),flush=True)\n            if met:\n                score=met[\'acerto_fato\']+met[\'auc_respondivel\']\n                # Seleção por métrica conjunta de desenvolvimento; ganho de ranking\n                # não autoriza regressão na separação de perguntas sem resposta.\n                good=all(met[k]>=baseline[k] for k in (\'acerto_fato\',\'auc_respondivel\'))\n                if good and score>best:\n                    best=score;chosen_step=step\n                    selected={k:v.detach().clone() for k,v in m.state_dict().items() if k in selected}\n                history.append({\'passo\':step,\'metricas\':met})\n    if dev:m.load_state_dict(selected,strict=False)\n    return chosen_step,history\n\ndef main():\n    torch.set_num_threads(2);torch.manual_seed(20261008)\n    from crivo import Crivo\n    comp=Crivo().compositor\n    cases=json.loads((ROOT/\'dados/leitura_ficha_tutor.json\').read_text())[\'casos\']\n    cases=[c for c in cases if c[\'assunto\'] in comp.itens]\n    origin=ROOT/\'experimentos/pesos_base/leitor_transformer\'\n    base,bpe,meta=carregar(origin,\'cpu\');base.eval()\n    path=OUT/\'cache_seis_camadas.pt\'\n    if path.exists():cache=torch.load(path,weights_only=True)\n    else:\n        cache=[]\n        with torch.no_grad():\n            for i,c in enumerate(cases):\n                facts=[f[\'texto\'] for f in comp.itens[c[\'assunto\']][\'fatos\']]\n                ids,u=lote_tensores([(c[\'pergunta\'],f) for f in facts],bpe,256,\'cpu\')\n                h=base.base.embedding(ids)+base.base.posicao(torch.arange(ids.shape[1]))\n                for block in base.base.blocos[:-2]:h=block(h)\n                cache.append((h.detach(),u))\n                if i%50==0:print(\'cache inferior\',i,flush=True)\n        torch.save(cache,path)\n    subjects=sorted({c[\'assunto\'] for c in cases});random.Random(20261008).shuffle(subjects)\n    fold={s:i%5 for i,s in enumerate(subjects)}\n    pred0=predizer(base,cache,list(range(len(cases))))\n    predictions=[None]*len(cases);selections=[]\n    for outer in range(5):\n        allowed=[s for s in subjects if fold[s]!=outer]\n        # Dev por assunto fixado antes do treino, cerca de 20% dos assuntos permitidos.\n        dev_subjects=set(allowed[::5])\n        train=[i for i,c in enumerate(cases) if fold[c[\'assunto\']]!=outer and c[\'assunto\'] not in dev_subjects]\n        dev=[i for i,c in enumerate(cases) if c[\'assunto\'] in dev_subjects]\n        test=[i for i,c in enumerate(cases) if fold[c[\'assunto\']]==outer]\n        m=copy.deepcopy(base)\n        print(\'BLOCO\',outer,\'treino\',len(train),\'dev\',len(dev),\'externo\',len(test),flush=True)\n        step,history=treinar(m,cache,cases,train,dev)\n        scores=predizer(m,cache,test)\n        for i,p in zip(test,scores):predictions[i]=p\n        selections.append({\'bloco\':outer,\'passo_escolhido\':step,\'historico_dev\':history})\n        print(\'resultado externo\',outer,metricas(scores,[cases[i] for i in test]),flush=True)\n        del m\n    report={\'protocolo\':\'Desenvolvimento em cinco blocos por assunto; parada escolhida em dev interno por assunto; dados humanos tutor usados como supervisão; congelados não usados\',\'baseline\':metricas(pred0,cases),\'validacao_externa\':metricas(predictions,cases),\'selecao\':selections,\'aprovado\':False}\n    (OUT/\'controle_duas_camadas.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\')\n    (OUT/\'predicoes_duas_camadas.json\').write_text(json.dumps([{\'assunto\':c[\'assunto\'],\'alvo\':c[\'fato\'],\'baseline\':a.tolist(),\'candidato\':b.tolist()} for c,a,b in zip(cases,pred0,predictions)]))\n    print(json.dumps(report,ensure_ascii=False),flush=True)\n    horizon=int(np.median([s[\'passo_escolhido\'] for s in selections]))\n    if horizon:\n        m=copy.deepcopy(base);treinar(m,cache,cases,list(range(len(cases))),[],horizon)\n        exportar(m,origin,OUT/\'candidato_duas_camadas\',meta,horizon,report[\'validacao_externa\'],report[\'baseline\'])\n        mp=OUT/\'candidato_duas_camadas/meta.json\';metadata=json.loads(mp.read_text())\n        metadata[\'adaptacao_revisada\']={\'dados\':\'perguntas humanas tutor; supervisão com validação agrupada por assunto\',\'controle\':report,\'horizonte_medido\':horizon}\n        mp.write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+\'\\n\')\n\nif __name__==\'__main__\':main()\n'}
CODIGO=OUT/'codigo'
CODIGO.mkdir(exist_ok=True)
for nome,conteudo in CODIGOS.items():
    (CODIGO/nome).write_text(conteudo,encoding='utf-8')
sys.path[:0]=[str(ROOT),str(ROOT/'scripts'),str(CODIGO)]


In [ ]:
CABECA_PRONTA={'cabeca.weight': [[0.030533699318766594, 0.009027492254972458, -0.09050823003053665, 0.06163599714636803, -0.07381341606378555, -0.03309377655386925, -0.059548404067754745, -0.06223414093255997, -0.010067088529467583, -0.05706571415066719, -0.06709562242031097, -0.05003978684544563, 0.006623414810746908, -0.00466562993824482, -0.10450433194637299, -0.06651340425014496, -0.11040252447128296, 0.032537512481212616, 0.04636342078447342, 0.03908563777804375, -0.01565636321902275, -0.032645270228385925, 0.011092395521700382, 0.03656153380870819, -0.060371555387973785, 0.021121928468346596, 0.06263525038957596, -0.05161187797784805, 0.056333884596824646, -0.04687318578362465, 0.09325919300317764, 0.048784591257572174, 0.025962019339203835, -0.02506832964718342, -0.03729754686355591, 0.023144176229834557, 0.07235646992921829, 0.02310197800397873, 0.043158918619155884, -0.024969078600406647, 0.00832187570631504, 0.11934403330087662, -0.04446401819586754, 0.042012523859739304, 0.03369313105940819, 0.05126173794269562, -0.015033897012472153, 0.022014092653989792, 0.06392278522253036, 0.006976878736168146, -0.0344969779253006, -0.04780041426420212, -0.10335678607225418, -0.005856889300048351, 0.047650955617427826, -0.14680765569210052, 0.0024430176708847284, -0.033374667167663574, -0.010395991615951061, 0.05767292529344559, -0.054639000445604324, -0.01516228262335062, 0.10485465824604034, -0.12861007452011108, 0.0075571611523628235, -0.05398548021912575, 0.06767603009939194, -0.04102242365479469, 0.057254236191511154, -0.03627939894795418, -0.04039640352129936, -0.019868776202201843, 0.08346005529165268, -0.07118871808052063, -0.020207982510328293, 0.08314468711614609, -0.01033549103885889, -0.007688149809837341, -0.005805176217108965, 0.0975351557135582, -0.032851409167051315, 0.013218497857451439, -0.10329467803239822, 0.03837794438004494, -0.007014311384409666, 0.08106787502765656, -0.0539301373064518, 0.061764590442180634, -0.03856696933507919, 0.07682543247938156, 0.03264688327908516, -0.060897160321474075, -0.0333433523774147, 0.024213869124650955, -0.056842926889657974, 0.02876432240009308, 0.024204114452004433, -0.036100246012210846, 0.0806412622332573, -0.045465003699064255, 0.03468029946088791, -0.04016018286347389, 0.008001605980098248, 0.053056277334690094, -0.015720190480351448, -0.07242605835199356, 0.020512480288743973, -0.09553417563438416, -0.022765297442674637, 0.09425769001245499, 0.01783239096403122, -0.013178741559386253, 0.06425638496875763, 0.04244328290224075, 0.04533765837550163, 0.04701492562890053, -0.002436470938846469, 0.04644922539591789, -0.08396783471107483, -0.034425344318151474, -0.04159396141767502, -0.06005234271287918, 0.06386660784482956, 0.0839661955833435, -0.035042472183704376, 0.006910609547048807, -0.009816346690058708, 0.03295407444238663, 0.10228625684976578, -0.0665624737739563, 0.025380995124578476, 0.001574253779835999, -0.005706205498427153, 0.019052457064390182, 0.050347864627838135, 0.023655619472265244, -0.03510470688343048, 0.03435911238193512, -0.009291870519518852, 0.0512695349752903, 0.059547219425439835, -0.025858957320451736, -0.026961520314216614, 0.09342965483665466, 0.06601741164922714, -0.07009442150592804, -0.03772899508476257, -0.12986674904823303, 0.012957371771335602, 0.06392740458250046, -0.08126889914274216, 0.02424926683306694, 0.005850763060152531, 0.0016680116532370448, -0.13596053421497345, 0.009220927022397518, -0.04926236718893051, 0.05730387568473816, 0.01889064349234104, -0.022104984149336815, -0.07976008951663971, 0.1313103884458542, -0.07399623095989227, -0.06401310861110687, -0.06389234960079193, 0.029901446774601936, 0.1052003875374794, -0.006050372961908579, 0.0755167305469513, -0.013568812981247902, 0.0010111852316185832, -0.04587048664689064, 0.02162882313132286, -0.10389652848243713, -0.0076944357715547085, -0.019063761457800865, -1.2742650142172351e-05, 0.027544239535927773, 0.11058585345745087, -0.10879842191934586, -0.0034727768506854773, 0.07254740595817566, 0.06421263515949249, -0.0025664663407951593, -0.050917770713567734, -0.07939649373292923, 0.0028235402423888445, 0.0617566853761673, 0.010466504842042923, 0.1104220598936081, 0.017674608156085014, -0.06680846214294434, 0.10702721774578094, 0.0651279091835022, 0.005724356975406408, 0.05018444359302521, -0.03467186912894249, 0.009307628497481346, 0.04540460929274559, -0.028397686779499054, 0.010382183827459812, -0.1585020124912262, 0.05162294581532478, 0.005322305019944906, -0.014978650957345963, -0.00475667929276824, 0.0021582988556474447, 0.06327421218156815, -0.06904570013284683, 0.0554356649518013, -0.07121332734823227, -0.0234298724681139, -0.07070700079202652, -0.07292510569095612, 0.032722100615501404, 0.13310329616069794, -0.047429412603378296, -0.03582169488072395, -0.041273083537817, -0.02924398146569729, -0.005935182329267263, 0.03235970810055733, 0.012882394716143608, -0.026673240587115288, -0.041454412043094635, -0.09037701785564423, 0.022562995553016663, -0.08694807440042496, -0.10769578814506531, 0.013200759887695312, -0.07321999222040176, 0.09230521321296692, 0.06458397209644318, 0.0002458370290696621, -0.026856306940317154, -0.015738528221845627, 0.029958590865135193, -0.05517926067113876, -0.05573754757642746, -0.045785341411828995, -0.05861431360244751, 0.0019655025098472834, 0.06596019864082336, -0.07008623331785202, 0.02745271846652031, 0.023679671809077263, 0.004572967998683453, -0.008714304305613041, 0.07386475801467896, 0.011655015870928764, 0.05543399602174759, 0.04192867502570152, 0.07289568334817886, -0.012146132066845894, -0.04045681282877922, 0.024899883195757866, -0.005933814682066441, 0.010066858492791653, -0.0516846738755703, 0.01341602299362421, 0.061319369822740555, -0.056549474596977234, -0.017460331320762634, 0.11610107123851776, -0.056588515639305115, 0.06432157009840012, -0.04172937199473381, -0.03955258056521416, 0.044112708419561386, -0.04287431761622429, -0.015083244070410728, -0.06683436781167984, 0.05757668986916542, 0.07048366218805313, -0.00998627208173275, 0.01465815119445324, 0.031840160489082336, 0.013533568941056728, 0.05998437479138374, 0.0538625568151474, 0.079243004322052, 0.06429505348205566, -0.04066041111946106, -0.03465074673295021, -0.05032506212592125, -0.015660734847187996, -0.11824002116918564, 0.0022497172467410564, -0.01625836081802845, -0.052944015711545944, -0.00763474777340889, -0.026108907535672188, -0.025322509929537773, 0.026307808235287666, 0.04938904941082001, 0.03141030669212341, -0.014856589026749134, -0.04342254623770714, 0.01972440630197525, -0.0985008180141449, -0.07736153155565262, -0.02129213698208332, 0.010882064700126648, 0.027540583163499832, -0.03222179785370827, 0.04631282761693001, 0.06710951030254364, -0.10440997034311295, -0.02619040198624134, -0.05151320621371269, 0.05089550465345383, 0.015316587872803211, -0.0544041246175766, -0.03974784165620804, 0.06259267032146454, 0.08905551582574844, -0.08785472810268402, -0.006238548085093498, 0.030058125033974648, 0.00788730289787054, -0.0047243814915418625, 0.045501936227083206, 0.016366278752684593, 0.05277419835329056, 0.13435593247413635, 0.007697920314967632, 0.01397742424160242, 0.013733694329857826, 0.06014168635010719, 0.021853944286704063, 0.06215725839138031, -0.03605436906218529, 0.00894663855433464, -0.023899737745523453, 0.014583444222807884, 0.0009756817016750574, -0.03205118700861931, -0.049123749136924744, 0.05560637265443802, 0.009544948115944862, 0.04868900030851364, 0.011915506795048714, 0.061093512922525406, 0.06339828670024872, 0.05256117507815361, 0.03195551037788391, -0.10259988903999329, -0.038749393075704575, -0.030130241066217422, 0.06352549046278, 0.002373459981754422, -0.03637487441301346, 0.05202626436948776, 0.055107202380895615, 0.014550830237567425, -0.05121762305498123, -0.059177614748477936, 0.08040724694728851, 0.05344429239630699, -0.09436041116714478, -0.09544163942337036, 0.014630526304244995, -0.0012094356352463365, 0.13696669042110443, -0.042172931134700775, 0.021800078451633453, 0.09536869823932648, -0.06874620169401169, 0.007168289739638567, 0.09181314706802368, 0.011800904758274555, 0.053576670587062836, -0.056656364351511, 0.06025499477982521, -0.005206202156841755, 0.08432404696941376, 0.07203677296638489, -0.011026270687580109, 0.05421203002333641, -0.09274822473526001, 0.07838249206542969, 0.04581679031252861, -0.05800306424498558, -0.008489783853292465]], 'cabeca.bias': [0.005714353173971176]}
RELATORIOS_MEDIDOS={'controle_cabeca.json': {'protocolo': '5 blocos externos por assunto; seleção L2 em 4 blocos internos; somente cabeça linear adaptada; tutor agora supervisão; testes congelados não usados', 'baseline': {'acerto_fato': 0.5527950310559007, 'auc_respondivel': 0.713999044433827, 'certos': 89, 'respondiveis': 161, 'sem_resposta': 65}, 'validacao_externa': {'acerto_fato': 0.6024844720496895, 'auc_respondivel': 0.757190635451505, 'certos': 97, 'respondiveis': 161, 'sem_resposta': 65}, 'selecao': [{'bloco': 0, 'l2': 0.03, 'validacao_interna': {'acerto_fato': 0.656, 'auc_respondivel': 0.7675294117647059, 'certos': 82, 'respondiveis': 125, 'sem_resposta': 51}, 'assuntos_externos': ['mundo_galileu', 'mundo_marie_curie', 'mundo_memoria', 'mundo_princesa_isabel', 'mundo_malala', 'mundo_dualidade_onda_particula', 'mundo_mutacao', 'mundo_efeito_estufa', 'mundo_estados_da_materia', 'mundo_memorias_postumas', 'mundo_pantanal', 'mundo_china_imperial', 'mundo_amazonia']}, {'bloco': 1, 'l2': 0.1, 'validacao_interna': {'acerto_fato': 0.6153846153846154, 'auc_respondivel': 0.7136094674556213, 'certos': 80, 'respondiveis': 130, 'sem_resposta': 52}, 'assuntos_externos': ['mundo_constituicao_1988', 'mundo_codigo_genetico', 'mundo_neuroplasticidade', 'mundo_ceticismo', 'mundo_dogma_central', 'mundo_luta_classes', 'mundo_logica_dedutiva', 'mundo_ansiedade', 'mundo_independencia_eua', 'mundo_colisao', 'mundo_replicacao_dna', 'mundo_fato_social', 'mundo_filosofia']}, {'bloco': 2, 'l2': 0.1, 'validacao_interna': {'acerto_fato': 0.6, 'auc_respondivel': 0.7520361990950226, 'certos': 78, 'respondiveis': 130, 'sem_resposta': 51}, 'assuntos_externos': ['mundo_pre_historia', 'mundo_urbanizacao_brasil', 'mundo_mercado', 'mundo_holocausto', 'mundo_inducao_eletromagnetica', 'mundo_adesao_lagartixa', 'mundo_onu', 'rede_neural', 'mundo_inercia', 'mundo_inflacao', 'mundo_fracao', 'mundo_difracao']}, {'bloco': 3, 'l2': 0.1, 'validacao_interna': {'acerto_fato': 0.5846153846153846, 'auc_respondivel': 0.7966618287373004, 'certos': 76, 'respondiveis': 130, 'sem_resposta': 53}, 'assuntos_externos': ['mundo_populacao_mundial', 'mundo_relatividade_geral', 'mundo_isaac_newton', 'mundo_idealismo_transcendental', 'mundo_mnemonica', 'mundo_revolucao_cientifica', 'mundo_mil_e_uma_noites', 'vacuo', 'mundo_alexandre_magno', 'mundo_socializacao', 'mundo_equacao_schrodinger', 'mundo_estresse']}, {'bloco': 4, 'l2': 0.03, 'validacao_interna': {'acerto_fato': 0.6201550387596899, 'auc_respondivel': 0.7754863244112915, 'certos': 80, 'respondiveis': 129, 'sem_resposta': 53}, 'assuntos_externos': ['mundo_spin', 'mundo_segunda_guerra', 'mundo_quark', 'mundo_falseabilidade', 'ser_humano', 'mundo_literatura_de_cordel', 'mundo_sono_rem', 'navegador', 'mundo_ligacao_quimica', 'mundo_sono', 'mundo_porcentagem', 'mundo_jornada_do_heroi']}], 'origem_sha256': '603ff3b6765d9fc82ff53517560ba3f1abca595e27f5a98c55911f1bb9a2b6dc', 'aprovado': False, 'pares_acerto': {'ganhos': 26, 'perdas': 18}, 'corpus_sha256': '0ce87aafd46a9d1ce23445056b8e900300b6fd8312b9684c4ac4d66c54f77c54', 'fontes': {'linguagem_profunda.py': '6654fc3e93c0d3d9adb1fa47e05fbdcd603a79beb4aac589e08b56c5c4b5c80c', 'leitor_transformer.py': '4ff88ec03cb44e0a4ef486f8e97ec6efa416d50eb1b304e874b3bfa5e3bc9871', 'experimentos/pesos_base/leitor_transformer/tokenizer.json': '6e37a7508def06d31f648a1fb816a67df99e806d450051b611a2406f044dbe7e'}, 'execucao': {'torch': '2.14.1+cpu', 'numpy': '2.5.3', 'dispositivo': 'cpu', 'semente': 20261008}}, 'controle_pipeline.json': {'sem': {'limiar': 0.5, 'limiar_aproximar': 0.5, 'validacao_cruzada_por_assunto': {'modelo_afirma': {'n': 40, 'certos': 39, 'errados': 1, 'nulos': 0}, 'modelo_aproxima': {'n': 55, 'certos': 39, 'errados': 8, 'nulos': 8}, 'modelo_cala': {'n': 94, 'certos': 39, 'errados': 11, 'nulos': 44}}, 'controle': {'aprovado': False}}, 'original': {'limiar': 0.5, 'limiar_aproximar': 0.5, 'validacao_cruzada_por_assunto': {'modelo_afirma': {'n': 40, 'certos': 39, 'errados': 1, 'nulos': 0}, 'modelo_aproxima': {'n': 55, 'certos': 39, 'errados': 8, 'nulos': 8}, 'modelo_cala': {'n': 94, 'certos': 39, 'errados': 11, 'nulos': 44}}, 'controle': {'aprovado': False}}, 'adaptado': {'limiar': 0.5, 'limiar_aproximar': 0.5, 'validacao_cruzada_por_assunto': {'modelo_afirma': {'n': 39, 'certos': 39, 'errados': 0, 'nulos': 0}, 'modelo_aproxima': {'n': 46, 'certos': 37, 'errados': 6, 'nulos': 3}, 'modelo_cala': {'n': 104, 'certos': 43, 'errados': 12, 'nulos': 49}}, 'controle': {'aprovado': False}}}, 'controle_congelado.json': {'protocolo': 'Pesos congelados antes de abrir controles; avaliação só em agregados; não usados para otimização', 'sha256': {'original': '603ff3b6765d9fc82ff53517560ba3f1abca595e27f5a98c55911f1bb9a2b6dc', 'cabeca': '34a1d7e1a1e0170e9104d9459d65daf0a6dae7a8a6e7a0cdece4009c79fbc516', 'duas_camadas': 'f1a4b516913d5d089fb3c27ac025cb6c7ee02e250f4d36304830d5e281e1a157'}, 'conjuntos': {'v1': {'casos': 72, 'avaliados': 72, 'metricas': {'original': {'acerto_fato': 0.44, 'auc_respondivel': 0.509090909090909, 'certos': 22, 'respondiveis': 50, 'sem_resposta': 22}, 'cabeca': {'acerto_fato': 0.6, 'auc_respondivel': 0.6590909090909091, 'certos': 30, 'respondiveis': 50, 'sem_resposta': 22}, 'duas_camadas': {'acerto_fato': 0.7, 'auc_respondivel': 0.6990909090909091, 'certos': 35, 'respondiveis': 50, 'sem_resposta': 22}}}, 'v2': {'casos': 70, 'avaliados': 70, 'metricas': {'original': {'acerto_fato': 0.3541666666666667, 'auc_respondivel': 0.5729166666666666, 'certos': 17, 'respondiveis': 48, 'sem_resposta': 22}, 'cabeca': {'acerto_fato': 0.5208333333333334, 'auc_respondivel': 0.7452651515151515, 'certos': 25, 'respondiveis': 48, 'sem_resposta': 22}, 'duas_camadas': {'acerto_fato': 0.4375, 'auc_respondivel': 0.7017045454545454, 'certos': 21, 'respondiveis': 48, 'sem_resposta': 22}}}}, 'aprovado_chat': False}, 'controle_chat.json': {'protocolo': 'Chat factual sem geração, classificadores originais, conjuntos congelados somente em agregados; instalação experimental em memória', 'aprovado': False, 'modos': {'original': {'teste': {'conjunto': 'teste', 'leitura': True, 'casos': 72, 'respondiveis': {'recusou': 16, 'aproximou_certo': 10, 'afirmou_certo': 23, 'afirmou_errado': 1}, 'sem_resposta': {'recusou': 15, 'afirmou': 3, 'ficha_posterior': 1, 'aproximou': 3}}, 'teste_v2': {'conjunto': 'teste_v2', 'leitura': True, 'casos': 70, 'respondiveis': {'recusou': 24, 'afirmou_certo': 12, 'aproximou_certo': 10, 'afirmou_errado': 2}, 'sem_resposta': {'afirmou': 4, 'recusou': 18}}}, 'adaptado': {'teste': {'conjunto': 'teste', 'leitura': True, 'casos': 72, 'respondiveis': {'recusou': 16, 'aproximou_certo': 10, 'afirmou_certo': 23, 'afirmou_errado': 1}, 'sem_resposta': {'recusou': 15, 'afirmou': 3, 'ficha_posterior': 1, 'aproximou': 3}}, 'teste_v2': {'conjunto': 'teste_v2', 'leitura': True, 'casos': 70, 'respondiveis': {'recusou': 25, 'afirmou_certo': 12, 'aproximou_certo': 9, 'afirmou_errado': 2}, 'sem_resposta': {'afirmou': 4, 'recusou': 18}}}}}}
META_PRONTA={'versao': 1, 'papel': 'leitor: probabilidade de um fato da ficha responder à pergunta', 'base': {'pasta': '/content/drive/MyDrive/CRIVO/transformer-leitor/base', 'pesos_sha256': '16102a72b21031b8bee2b4edfa633a359f7f08c5beab552b05f294e656b0cc67', 'config': {'vocabulario': 8192, 'dimensao': 384, 'camadas': 8, 'cabecas': 8, 'contexto': 256, 'dropout': 0.1}, 'passo_pretreino': 20000}, 'treino': {'passo': 1000, 'passos': 4000, 'lote': 64, 'lr': 5e-05, 'semente': 20261006, 'wikipedia': True, 'artigos': 60000, 'dados': 'sintéticos das fichas (sem tutor, testes v1/v2, teste sem nome e astronomia), com e sem o nome do assunto; opcionalmente exercícios da Wikipédia'}, 'validacao_tutor': {'acerto_fato': 0.6024844720496895, 'auc_respondivel': 0.757190635451505, 'certos': 97, 'respondiveis': 161, 'sem_resposta': 65}, 'controle': {'aprovado': False, 'criterio': 'Requer ganho end-to-end sem mais erros nos controles congelados.'}, 'adaptacao_cabeca': {'parametros_treinados': 385, 'l2': 0.1, 'dados': 'tutor como supervisão, validação cruzada aninhada por assunto; base totalmente congelada', 'controle': {'protocolo': '5 blocos externos por assunto; seleção L2 em 4 blocos internos; somente cabeça linear adaptada; tutor agora supervisão; testes congelados não usados', 'baseline': {'acerto_fato': 0.5527950310559007, 'auc_respondivel': 0.713999044433827, 'certos': 89, 'respondiveis': 161, 'sem_resposta': 65}, 'validacao_externa': {'acerto_fato': 0.6024844720496895, 'auc_respondivel': 0.757190635451505, 'certos': 97, 'respondiveis': 161, 'sem_resposta': 65}, 'selecao': [{'bloco': 0, 'l2': 0.03, 'validacao_interna': {'acerto_fato': 0.656, 'auc_respondivel': 0.7675294117647059, 'certos': 82, 'respondiveis': 125, 'sem_resposta': 51}, 'assuntos_externos': ['mundo_galileu', 'mundo_marie_curie', 'mundo_memoria', 'mundo_princesa_isabel', 'mundo_malala', 'mundo_dualidade_onda_particula', 'mundo_mutacao', 'mundo_efeito_estufa', 'mundo_estados_da_materia', 'mundo_memorias_postumas', 'mundo_pantanal', 'mundo_china_imperial', 'mundo_amazonia']}, {'bloco': 1, 'l2': 0.1, 'validacao_interna': {'acerto_fato': 0.6153846153846154, 'auc_respondivel': 0.7136094674556213, 'certos': 80, 'respondiveis': 130, 'sem_resposta': 52}, 'assuntos_externos': ['mundo_constituicao_1988', 'mundo_codigo_genetico', 'mundo_neuroplasticidade', 'mundo_ceticismo', 'mundo_dogma_central', 'mundo_luta_classes', 'mundo_logica_dedutiva', 'mundo_ansiedade', 'mundo_independencia_eua', 'mundo_colisao', 'mundo_replicacao_dna', 'mundo_fato_social', 'mundo_filosofia']}, {'bloco': 2, 'l2': 0.1, 'validacao_interna': {'acerto_fato': 0.6, 'auc_respondivel': 0.7520361990950226, 'certos': 78, 'respondiveis': 130, 'sem_resposta': 51}, 'assuntos_externos': ['mundo_pre_historia', 'mundo_urbanizacao_brasil', 'mundo_mercado', 'mundo_holocausto', 'mundo_inducao_eletromagnetica', 'mundo_adesao_lagartixa', 'mundo_onu', 'rede_neural', 'mundo_inercia', 'mundo_inflacao', 'mundo_fracao', 'mundo_difracao']}, {'bloco': 3, 'l2': 0.1, 'validacao_interna': {'acerto_fato': 0.5846153846153846, 'auc_respondivel': 0.7966618287373004, 'certos': 76, 'respondiveis': 130, 'sem_resposta': 53}, 'assuntos_externos': ['mundo_populacao_mundial', 'mundo_relatividade_geral', 'mundo_isaac_newton', 'mundo_idealismo_transcendental', 'mundo_mnemonica', 'mundo_revolucao_cientifica', 'mundo_mil_e_uma_noites', 'vacuo', 'mundo_alexandre_magno', 'mundo_socializacao', 'mundo_equacao_schrodinger', 'mundo_estresse']}, {'bloco': 4, 'l2': 0.03, 'validacao_interna': {'acerto_fato': 0.6201550387596899, 'auc_respondivel': 0.7754863244112915, 'certos': 80, 'respondiveis': 129, 'sem_resposta': 53}, 'assuntos_externos': ['mundo_spin', 'mundo_segunda_guerra', 'mundo_quark', 'mundo_falseabilidade', 'ser_humano', 'mundo_literatura_de_cordel', 'mundo_sono_rem', 'navegador', 'mundo_ligacao_quimica', 'mundo_sono', 'mundo_porcentagem', 'mundo_jornada_do_heroi']}], 'origem_sha256': '603ff3b6765d9fc82ff53517560ba3f1abca595e27f5a98c55911f1bb9a2b6dc', 'aprovado': False, 'pares_acerto': {'ganhos': 26, 'perdas': 18}, 'corpus_sha256': '0ce87aafd46a9d1ce23445056b8e900300b6fd8312b9684c4ac4d66c54f77c54', 'fontes': {'linguagem_profunda.py': '6654fc3e93c0d3d9adb1fa47e05fbdcd603a79beb4aac589e08b56c5c4b5c80c', 'leitor_transformer.py': '4ff88ec03cb44e0a4ef486f8e97ec6efa416d50eb1b304e874b3bfa5e3bc9871', 'experimentos/pesos_base/leitor_transformer/tokenizer.json': '6e37a7508def06d31f648a1fb816a67df99e806d450051b611a2406f044dbe7e'}, 'execucao': {'torch': '2.14.1+cpu', 'numpy': '2.5.3', 'dispositivo': 'cpu', 'semente': 20261008}}}, 'controle_congelado_leitor': {'protocolo': 'Pesos congelados antes de abrir controles; avaliação só em agregados; não usados para otimização', 'sha256': {'original': '603ff3b6765d9fc82ff53517560ba3f1abca595e27f5a98c55911f1bb9a2b6dc', 'cabeca': '34a1d7e1a1e0170e9104d9459d65daf0a6dae7a8a6e7a0cdece4009c79fbc516', 'duas_camadas': 'f1a4b516913d5d089fb3c27ac025cb6c7ee02e250f4d36304830d5e281e1a157'}, 'conjuntos': {'v1': {'casos': 72, 'avaliados': 72, 'metricas': {'original': {'acerto_fato': 0.44, 'auc_respondivel': 0.509090909090909, 'certos': 22, 'respondiveis': 50, 'sem_resposta': 22}, 'cabeca': {'acerto_fato': 0.6, 'auc_respondivel': 0.6590909090909091, 'certos': 30, 'respondiveis': 50, 'sem_resposta': 22}, 'duas_camadas': {'acerto_fato': 0.7, 'auc_respondivel': 0.6990909090909091, 'certos': 35, 'respondiveis': 50, 'sem_resposta': 22}}}, 'v2': {'casos': 70, 'avaliados': 70, 'metricas': {'original': {'acerto_fato': 0.3541666666666667, 'auc_respondivel': 0.5729166666666666, 'certos': 17, 'respondiveis': 48, 'sem_resposta': 22}, 'cabeca': {'acerto_fato': 0.5208333333333334, 'auc_respondivel': 0.7452651515151515, 'certos': 25, 'respondiveis': 48, 'sem_resposta': 22}, 'duas_camadas': {'acerto_fato': 0.4375, 'auc_respondivel': 0.7017045454545454, 'certos': 21, 'respondiveis': 48, 'sem_resposta': 22}}}}, 'aprovado_chat': False}}
if MODO == 'pesos_prontos':
    import torch
    sys.path.insert(0,str(CODIGO))
    from continuar_leitor import carregar, exportar
    modelo,bpe,meta_original=carregar(ORIGEM,'cpu')
    with torch.no_grad():
        modelo.cabeca.weight.copy_(torch.tensor(CABECA_PRONTA['cabeca.weight']))
        modelo.cabeca.bias.copy_(torch.tensor(CABECA_PRONTA['cabeca.bias']))
    controle=RELATORIOS_MEDIDOS['controle_cabeca.json']
    exportar(modelo,ORIGEM,OUT/'candidato_cabeca',meta_original,0,controle['validacao_externa'],controle['baseline'])
    (OUT/'candidato_cabeca/meta.json').write_text(json.dumps(META_PRONTA,ensure_ascii=False,indent=2)+'\n')
    for nome,relatorio in RELATORIOS_MEDIDOS.items():
        (OUT/nome).write_text(json.dumps(relatorio,ensure_ascii=False,indent=2)+'\n')
    print('Novos pesos prontos. Os relatórios abaixo vêm do experimento já executado; esta célula não fez novo treino.')


## 2. Reproduzir o ajuste e validar por assunto

Não precisa repetir o treino anterior de 1.000 passos. A base de 17 milhões fica congelada.


In [ ]:
if MODO == 'reproduzir_treino':
    for nome in ['diagnosticar_cabeca.py','controlar_pipeline.py']:
        subprocess.run([sys.executable,'-u',str(CODIGO/nome)],check=True)
r=json.loads((OUT/'controle_cabeca.json').read_text())
print('Validação por assunto:',r['baseline'],'→',r['validacao_externa'])
print('Este controle é desenvolvimento; não é o resultado do teste congelado.')


## 3. Medir o leitor e o Crivo completo

Esta etapa não treina nem aprova pesos. O relatório do chat informa se o ganho chega à resposta final.


In [ ]:
# Somente medição: nenhuma pergunta dos controles entra no treinamento.
if MODO == 'reproduzir_treino':
    for nome in ['avaliar_controles.py','avaliar_chat.py']:
        subprocess.run([sys.executable,'-u',str(CODIGO/nome)],check=True)
else:
    print('Modo pesos prontos: usando os relatórios do experimento já executado.')
print('Pesos permanecem experimentais; ganho isolado do leitor não aprova o chat.')


## 4. Guardar os pesos e todos os relatórios


In [ ]:
pacote=OUT/'leitor_revisado_v2.zip'
with zipfile.ZipFile(pacote,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in (OUT/'candidato_cabeca').iterdir():z.write(p,'leitor_transformer/'+p.name)
    for name in ['controle_cabeca.json','controle_pipeline.json','controle_congelado.json','controle_chat.json']:
        z.write(OUT/name,'relatorios/'+name)
print('Pesos e relatórios:',pacote)
from google.colab import files
files.download(str(pacote))
